# California Housing Price Prediction using XGBoost
This notebook implements an end-to-end regression model using **XGBoost Regressor** to predict California median house values based on socioeconomic and geographic variables from census data.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn import metrics

## 2. Load Dataset & Inspection

In [ ]:
data_housing = pd.read_csv('housing.csv')
data_housing.head()

In [ ]:
# Cek dimensi dataset dan ketersediaan data kosong
print('Ukuran Dataset:', data_housing.shape)
print('\nJumlah Missing Values:')
print(data_housing.isnull().sum())

## 3. Exploratory Data Analysis (Correlation Heatmap)

In [ ]:
# Menghitung matriks korelasi antar fitur
korelasi = data_housing.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(korelasi, cbar=True, square=True, fmt='.2f', annot=True, annot_kws={'size': 9}, cmap='Blues')
plt.title('Peta Korelasi Fitur terhadap Nilai Properti (MedHouseVal)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Feature Selection & Train-Test Split

In [ ]:
# Pisahkan fitur prediktor (X) dan target (y)
X = data_housing.drop(columns='MedHouseVal', axis=1)
y = data_housing['MedHouseVal']

# Pembagian 80% train dan 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

print('Total Sampel Data :', X.shape[0])
print('Sampel Data Latih :', X_train.shape[0])
print('Sampel Data Uji   :', X_test.shape[0])

## 5. Model Training (XGBoost Regressor)

In [ ]:
# Pelatihan model XGBoost
model = XGBRegressor(random_state=2)
model.fit(X_train, y_train)

## 6. Model Evaluation (R² Score & MAE)

In [ ]:
# Evaluasi pada data latih
prediksi_latih = model.predict(X_train)
r2_latih = metrics.r2_score(y_train, prediksi_latih)
mae_latih = metrics.mean_absolute_error(y_train, prediksi_latih)

# Evaluasi pada data uji
prediksi_uji = model.predict(X_test)
r2_uji = metrics.r2_score(y_test, prediksi_uji)
mae_uji = metrics.mean_absolute_error(y_test, prediksi_uji)

print('=== NILAI EVALUASI DATA LATIH ===')
print(f'R-squared Score         : {r2_latih:.4f}')
print(f'Mean Absolute Error (MAE): {mae_latih:.4f}')

print('\n=== NILAI EVALUASI DATA UJI ===')
print(f'R-squared Score         : {r2_uji:.4f}')
print(f'Mean Absolute Error (MAE): {mae_uji:.4f}')

## 7. Feature Importance Analysis
Menganalisis fitur apa saja yang memiliki bobot pengaruh paling besar dalam keputusan pohon XGBoost.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances.plot(kind='barh', color='#0284c7', edgecolor='black', linewidth=0.5)
plt.title('Tingkat Pengaruh Fitur (Feature Importance) - XGBoost', fontsize=13, fontweight='bold')
plt.xlabel('Relative Importance Score')
plt.ylabel('Fitur')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 8. Actual vs Predicted Price Visualization

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, prediksi_uji, color='#2563eb', alpha=0.3, edgecolors='none')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='#dc2626', linestyle='--', linewidth=2, label='Ideal Fit (y = x)')
plt.xlabel('Nilai Aktual ($100.000)')
plt.ylabel('Prediksi Model ($100.000)')
plt.title('Sebaran Nilai Aktual vs Prediksi XGBoost', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 9. New Property Price Estimation (Inference)

In [ ]:
# Format input: (MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude)
sample_rumah = (8.3252, 41.0, 6.984127, 1.023810, 322.0, 2.555556, 37.88, -122.23)

sample_array = np.asarray(sample_rumah).reshape(1, -1)
taksiran = model.predict(sample_array)[0]

harga_usd = taksiran * 100000
harga_idr = harga_usd * 16000

print('=== HASIL ESTIMASI NILAI PROPERTI ===')
print(f'Skor Prediksi Model : {taksiran:.3f}')
print(f'Estimasi Harga USD  : ${harga_usd:,.2f} USD')
print(f'Estimasi Harga IDR  : Rp {harga_idr:,.2f} (Kurs Rp16.000)')